# Real-time pipeline on every clip, on a Kaggle GPU

Runs `realtime_pipeline.py` on every video of the attached dataset with the GPU
settings: every frame processed, the lost ball searched on all the court tiles,
and the colour fallback by the players. Each clip is calibrated again first (its
own court fit and background), so it is tracked from its first frame.

The videos must be clips from one fixed camera (one rally or a few, no cuts):
cut highlight videos into such clips before adding them to the dataset.

The results come out as **`realtime_outputs.zip`**, with the same folders as the
project (`outputs/realtime/<clip>/...`): unzip it in the project folder on your PC
and watch a clip with

    python live_view.py --video data/<clip>.mp4 --realtime

The ball CSVs alone are also in `ball/`, and the overlay videos of the pipeline
(Kalman estimate, model crops, events) in `overlay/`.

**Setup.** A private dataset with the videos, `ball_motion.pt` (the motion
detector) and `ball_tracker.pt` (the offline YOLOv8x, for the fallback).
*Settings*: GPU (T4 or P100), Internet on. Then *Save Version -> Save & Run All*.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: set Settings -> Accelerator to a GPU'

## 2. Get the code

In [ ]:
BRANCH = 'realtime-motion-ball'
!git clone -q -b {BRANCH} https://github.com/marcomassa-unitn/tennis_players_tracking.git
%cd tennis_players_tracking
!pip -q install ultralytics

## 3. Find the videos and the weights
They are searched for in all the inputs, so the name of the dataset does not matter.

In [ ]:
import glob, os, shutil, json
files = glob.glob('/kaggle/input/**/*', recursive=True)
VIDEOS = sorted(f for f in files if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')))
MODEL = next((f for f in files if os.path.basename(f) == 'ball_motion.pt'), None)
FALLBACK = next((f for f in files if os.path.basename(f) == 'ball_tracker.pt'), None)
assert VIDEOS, 'No video found in the inputs'
assert MODEL, 'ball_motion.pt not found: add the motion detector weights to the dataset'
print('Model   :', MODEL)
print('Fallback:', FALLBACK or 'none (ball_tracker.pt not in the dataset)')
print(len(VIDEOS), 'videos:', *[os.path.basename(v) for v in VIDEOS], sep='\n  ')

## 4. Run every clip
Every frame, as on a GPU that keeps up with the video. Set `OVERLAY = False` to
skip the overlay videos.

In [ ]:
OVERLAY = True
fallback = f'--fallback "{FALLBACK}"' if FALLBACK else ''
failed = []
for v in VIDEOS:
    stem = os.path.splitext(os.path.basename(v))[0]
    overlay = f'--video-out "outputs/realtime/{stem}/overlay.mp4"' if OVERLAY else ''
    print('=' * 70, '\n', stem)
    !python realtime_pipeline.py --video "{v}" --model "{MODEL}" {fallback} --recalibrate --clock all --search tiles --search-every 1 {overlay} --log-every 300
    if not os.path.exists(f'outputs/realtime/{stem}/ball_coordinates/ball_{stem}.csv'):
        failed.append(stem)
print('Failed:', failed or 'none')

## 5. Summary

In [ ]:
import pandas as pd
rows = []
for path in sorted(glob.glob('outputs/realtime/*/realtime/summary.json')):
    s = json.load(open(path))
    rows.append({'clip': path.split('/')[2], 'frames': s['frames'],
                 'ball positions': s['ball_positions_final'], 'events': s['events'],
                 'shots': s['shots'], 'fps': s['steady_fps'], 'real time': s['real_time']})
pd.DataFrame(rows)

## 6. Save the results
Only `/kaggle/working` is kept. Download `realtime_outputs.zip` (everything), or
just the ball CSVs from `ball/` and the overlay videos from `overlay/`.

In [ ]:
out = '/kaggle/working'
os.makedirs(f'{out}/ball', exist_ok=True)
os.makedirs(f'{out}/overlay', exist_ok=True)
for csv_path in glob.glob('outputs/realtime/*/ball_coordinates/ball_*.csv'):
    if 'candidates' not in csv_path:
        shutil.copy(csv_path, f'{out}/ball/')
for mp4 in glob.glob('outputs/realtime/*/overlay.mp4'):
    shutil.copy(mp4, f"{out}/overlay/{mp4.split('/')[2]}_overlay.mp4")
# The zip keeps the project layout; the overlay videos are left out of it.
shutil.make_archive(f'{out}/realtime_outputs', 'zip', '.', 'outputs/realtime')
!cd /kaggle/working && zip -q -d realtime_outputs.zip "outputs/realtime/*/overlay.mp4" || true
print(sorted(os.listdir(out)))
%cd /kaggle/working
!rm -rf tennis_players_tracking